# 02_resources_prompts

Stage 2 of the `codes/mcp/` track: **resources** and **prompts** on the same
**FastMCP + Pydantic** server as stage 1's tools.

Notebook path: `codes/mcp/02_resources_prompts/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Install once (venv recommended), from this stage directory:

```bash
pip install -r ../../requirements.txt
```

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`Client(mcp) → list_resources / read_resource` and `list_prompts / get_prompt`

Static resources show up in `list_resources`. A URI with `{param}` is a
**template** (`list_resource_templates`); reading `note://alpha` fills `slug`.
Prompt arguments come from `Annotated` + `Field` — still no hand-rolled
JSON-RPC or schema dicts.

FastMCP's Client is async — cells below use top-level `await` (Jupyter /
IPython). From a plain script, wrap the same coroutine in `asyncio.run(...)`.

In [1]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "02_resources_prompts" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "02_resources_prompts"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/02_resources_prompts")

ROOT = lab_root()
print(ROOT)

/workspace/AEFS-Notes/codes/mcp/02_resources_prompts


## 1. Read a host-facing resources/list shape

`payload` is what a host sees after listing resources — URIs plus mime types,
**not** a hand-built JSON-RPC envelope.

Print every resource `uri`, then print the `mimeType` of `lab://greeting`.

In [2]:
payload = {
    "resources": [
        {
            "uri": "lab://greeting",
            "name": "greeting",
            "description": "Static lab greeting (no URI parameters).",
            "mimeType": "text/plain",
        }
    ],
    "resourceTemplates": [
        {
            "uriTemplate": "note://{slug}",
            "name": "note",
            "description": "Templated note: clients read note://alpha, …",
            "mimeType": "text/markdown",
        }
    ],
}

# answer

uris = [r["uri"] for r in payload["resources"]]
print("uris:", uris)
greeting = next(r for r in payload["resources"] if r["uri"] == "lab://greeting")
print("greeting.mimeType:", greeting["mimeType"])

uris: ['lab://greeting']
greeting.mimeType: text/plain


## 2. Static resource + URI template on one FastMCP server

Create a tiny `FastMCP("qa-res")` in this cell with:

1. `@server.resource("lab://ping", mime_type="text/plain")` → returns `"pong"`
2. `@server.resource("item://{item_id}", mime_type="application/json")` →
   returns a JSON string `{"id": "<item_id>"}`

Then with `Client(server)` print:

1. negotiated `protocol_version` (expect `2026-07-28`)
2. static resource URIs from `list_resources()`
3. template URI strings from `list_resource_templates()`
4. text from `read_resource("lab://ping")` and `read_resource("item://42")`

In [3]:
import json
from typing import Annotated

from fastmcp import Client, FastMCP
from pydantic import Field

# answer

server = FastMCP("qa-res")

@server.resource("lab://ping", mime_type="text/plain")
def ping() -> str:
    """Static ping resource."""
    return "pong"

@server.resource("item://{item_id}", mime_type="application/json")
def item(item_id: str) -> str:
    """Templated item by id."""
    return json.dumps({"id": item_id})

async with Client(server) as client:
    print("protocol:", client.protocol_version)
    resources = await client.list_resources()
    print("resources:", [str(r.uri) for r in resources])
    templates = await client.list_resource_templates()
    print("templates:", [t.uri_template for t in templates])
    ping_body = await client.read_resource("lab://ping")
    print("ping:", ping_body[0].text)
    item_body = await client.read_resource("item://42")
    print("item:", item_body[0].text)

protocol: 2026-07-28
resources: ['lab://ping']
templates: ['item://{item_id}']
ping: pong
item: {"id": "42"}


## 3. Drive the lab server: resources + prompts

Import `mcp` from `server.py` in this directory. With `Client(mcp)`:

1. print `protocol_version` and `server_info.name`
2. `list_resources` / `list_resource_templates` / `list_prompts` — print names/URIs
3. `read_resource("note://alpha")` and print the text
4. `get_prompt("critique", {"topic": "resources"})` and print the first message text

Same Client entry as stage 1 — only the MCP surface grows.

In [4]:
import sys

sys.path.insert(0, str(ROOT))
from fastmcp import Client
from server import mcp

# answer

async with Client(mcp) as client:
    print("protocol:", client.protocol_version)
    print("server:", client.server_info.name if client.server_info else None)
    resources = await client.list_resources()
    print("resources:", [str(r.uri) for r in resources])
    templates = await client.list_resource_templates()
    print("templates:", [t.uri_template for t in templates])
    prompts = await client.list_prompts()
    print("prompts:", [p.name for p in prompts])
    note = await client.read_resource("note://alpha")
    print("note:", note[0].text)
    result = await client.get_prompt("critique", {"topic": "resources"})
    print("prompt:", result.messages[0].content.text)

protocol: 2026-07-28
server: lab-resources-prompts
resources: ['lab://greeting']
templates: ['note://{slug}']
prompts: ['critique']
note: # note:alpha
body for alpha
prompt: Critique this topic in 3 bullets:
resources


## 4. Register a Pydantic-typed prompt

Build a local `FastMCP("qa-prompt")` that exposes
`summarize(text: Annotated[str, Field(...)]) -> str` via `@server.prompt`.
Call `get_prompt("summarize", {"text": "MCP stage 2"})` and print the rendered
user-message text.

Extension point for later stages: keep adding ordinary typed callables;
transport / protocol stay with FastMCP.

In [5]:
# answer

ext = FastMCP("qa-prompt")

@ext.prompt
def summarize(
    text: Annotated[str, Field(description="Text to summarize")],
) -> str:
    """Ask for a one-sentence summary."""
    return f"Summarize in one sentence:\n{text}"

async with Client(ext) as client:
    result = await client.get_prompt("summarize", {"text": "MCP stage 2"})
    print(result.messages[0].content.text)

Summarize in one sentence:
MCP stage 2
